# Airflow 1: Test a local DAG

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/airflow/airflow-01-test-a-dag.ipynb)

Run a real Airflow DAG with a temporary SQLite metadata database. Follow the [Airflow test-mode guide](https://airflow.apache.org/docs/apache-airflow/stable/executor/debug.html).

*Last reviewed: November 2026, against Apache Airflow 3.4.*

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q apache-airflow~=3.4

## Setup and sample data

Create an isolated metadata database and an importable DAG file. The task input represents a batch of two orders.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory()
root_dir = Path(workspace.name)
airflow_env = {
    **os.environ,
    "AIRFLOW_HOME": str(root_dir),
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
}
subprocess.run([sys.executable, "-m", "airflow", "db", "migrate"],
               env=airflow_env, check=True, capture_output=True, text=True)
print("Metadata database ready")

## Run TaskFlow dependencies in process

A task call in a DAG definition creates a dependency. Airflow runs the upstream task and delivers its small return value to the downstream task. dag.test executes a local run against the metadata database.

**Where it matters**

Check a TaskFlow DAG before deploying it to a scheduler.

**Watch out for**

Test mode does not exercise a production executor or scheduler. Pass paths or IDs between tasks when datasets are large.

In [ ]:
dag_file = root_dir / "dags" / "orders_dag.py"
dag_file.parent.mkdir(exist_ok=True)
dag_file.write_text("""
from datetime import datetime, timezone
from airflow.sdk import dag, task


@dag(schedule=None, start_date=datetime(2026, 1, 1, tzinfo=timezone.utc))
def orders_demo():
    @task
    def extract():
        return [1200, 800]

    @task
    def summarize(amounts):
        total = sum(amounts)
        print("Revenue in cents:", total)
        return total

    summarize(extract())


orders = orders_demo()

if __name__ == "__main__":
    run = orders.test()
    print("DAG state:", run.state)
""")
completed = subprocess.run([sys.executable, str(dag_file)], env=airflow_env,
                           check=True, capture_output=True, text=True)
for line in (completed.stdout + "\n" + completed.stderr).splitlines():
    if "Revenue in cents:" in line or "DAG state:" in line:
        print(line)

**What to notice:** The downstream task reports 2000 cents, and the DAG run reaches success.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Which task must succeed before summarize can run?

In [ ]:
# Try your solution here.

In [ ]:
print("extract")

## Key takeaways

- Check a TaskFlow DAG before deploying it to a scheduler.

**Next:** [Orchestration patterns](../concepts/orchestration-patterns.ipynb)